In [ ]:
import numpy as np
import pandas as pd
from pylab import plt, mpl
from sklearn.metrics import accuracy_score
import os
import papermill
import talib as ta
import optuna
from sklearn.model_selection import TimeSeriesSplit
from sklearn.neural_network import MLPClassifier
import tensorflow as tf
from keras.layers import Dense
from keras.models import Sequential
from sklearn.inspection import permutation_importance

# Frecuencia obtenida desde el main
try:
    print(f"Frecuencia recibida desde papermill: {frequency}")
except NameError:
    print(f"No se recibió 'frequency'.")


# Cargar los datos para esta frecuencia de un archivo creado por el main
file_name = f"processed_data_{frequency}_glob.csv"
data = pd.read_csv(file_name, index_col='timestamp')
data

,BTCUSDT,AVAXUSDT,XRPUSDT
timestamp,,,
2020-09-22,10529.61,5.3193,0.23302
2020-09-23,10241.46,3.5350,0.22164
2020-09-24,10736.32,4.6411,0.23276
2020-09-25,10686.67,4.7134,0.24154
2020-09-26,10728.60,4.5200,0.24153


Función para guardar los datos. Hace un archivo por cada frecuencia. Guarda en cada línea el modelo que se ha empleado, el activo, accuracy e in/out-sample.

In [ ]:
def save_results_global(model, crypto, acc, sample, frequency='1d'):
    file_name = f'accuracy_results_{frequency}_glob.csv'

    if os.path.exists(file_name):
        df_results = pd.read_csv(file_name)
    else:
        df_results = pd.DataFrame(columns=['Model', 'Asset', 'Accuracy', 'Vald/Test'])

    new_row = pd.DataFrame([[model, crypto, acc, sample]], columns=['Model', 'Asset', 'Accuracy', 'Vald/Test'])
    df_results = pd.concat([df_results, new_row], ignore_index=True)
    df_results.to_csv(file_name, index=False)

Creamos las características que usaremos para hacer el aprendizaje ahora y las retardamos.

In [ ]:
def charact_lags(data, ric, lags, window_pred, window=30):
    cols = []
    df = pd.DataFrame(data[ric])
    df.dropna(inplace=True)
    df['r'] = np.log(df / df.shift()) #retornos
    df['sma'] = df[ric].rolling(window).mean()  #media movil de la ventana
    df['min'] = df[ric].rolling(window).min() #mínimo de la ventana
    df['max'] = df[ric].rolling(window).max() #máximo de la ventana
    df['mom'] = df[ric].pct_change(window) #momentum de la ventana pct_change(12)
    df['vol'] = df['r'].rolling(window).std() #volatilidad de la ventana
    df['rsi'] = ta.RSI(df[ric], timeperiod=window) #rsi de la ventana
    df['atr'] = ta.ATR(df[ric], df[ric], df[ric], timeperiod=window) #atr de la ventana
    df.dropna(inplace=True)
    df = df.iloc[:-window_pred]
    df['d'] = np.where(df[ric].shift(-window_pred) > df[ric], 1, 0) # columna binaria, 0 si los precios bajarán, 1 si subirán
    print(df['d'].value_counts(normalize=True)) #comprueba si los datos están desbalanceados 
    features = [ric, 'r', 'sma', 'min', 'max', 'mom', 'vol', 'rsi', 'atr']
    for f in features:
        for lag in range(1, lags + 1):
            col = f'{f}_lag_{lag}'
            df[col] = df[f].shift(lag)
            cols.append(col)
    df.dropna(inplace=True)
    return df, cols

lags = 5

dfs = {}
for ric in data:
    df, cols = charact_lags(data, ric, lags, window_pred)
    dfs[ric] = df.dropna(), cols

d
1    0.535387
0    0.464613
Name: proportion, dtype: float64
d
0    0.515072
1    0.484928
Name: proportion, dtype: float64
d
0    0.52228
1    0.47772
Name: proportion, dtype: float64


Comentar que he mirado si los datos están desbalanceados 

In [25]:
dfs[ric][0]

,XRPUSDT,r,sma,min,max,mom,vol,rsi,atr,d,...,rsi_lag_1,rsi_lag_2,rsi_lag_3,rsi_lag_4,rsi_lag_5,atr_lag_1,atr_lag_2,atr_lag_3,atr_lag_4,atr_lag_5
timestamp,,,,,,,,,,,,,,,,,,,,,
2020-10-27,0.25273,0.018490,0.247623,0.23273,0.25709,0.038588,0.018629,57.099688,0.003870,0,...,55.317583,57.840898,59.440599,58.907397,60.127914,0.003843,0.003802,0.003828,0.003908,0.003961
2020-10-28,0.24540,-0.029432,0.247767,0.23273,0.25709,0.017962,0.019369,53.598651,0.003985,0,...,57.099688,55.317583,57.840898,59.440599,58.907397,0.003870,0.003843,0.003802,0.003828,0.003908
2020-10-29,0.24239,-0.012342,0.247752,0.23273,0.25709,-0.001894,0.019465,52.238035,0.003952,1,...,53.598651,57.099688,55.317583,57.840898,59.440599,0.003985,0.003870,0.003843,0.003802,0.003828
2020-10-30,0.23905,-0.013875,0.247663,0.23273,0.25709,-0.011005,0.019612,50.758935,0.003932,1,...,52.238035,53.598651,57.099688,55.317583,57.840898,0.003952,0.003985,0.003870,0.003843,0.003802
2020-10-31,0.23968,0.002632,0.247713,0.23273,0.25709,0.006256,0.019431,51.029495,0.003822,1,...,50.758935,52.238035,53.598651,57.099688,55.317583,0.003932,0.003952,0.003985,0.003870,0.003843
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-21,2.23800,-0.018286,2.147353,1.39840,2.72450,0.794707,0.078791,61.230947,0.087947,0,...,62.204662,61.629097,63.277124,69.778755,68.774720,0.089555,0.091254,0.091942,0.086250,0.086355
2024-12-22,2.20310,-0.015717,2.171703,1.39840,2.72450,0.496061,0.073957,60.404383,0.086179,0,...,61.230947,62.204662,61.629097,63.277124,69.778755,0.087947,0.089555,0.091254,0.091942,0.086250
2024-12-23,2.26170,0.026251,2.198143,1.39840,2.72450,0.540143,0.073927,61.311540,0.085259,0,...,60.404383,61.230947,62.204662,61.629097,63.277124,0.086179,0.087947,0.089555,0.091254,0.091942


In [26]:
# Lista de criptomonedas (clave en dfs)
cryptos = list(dfs.keys())

# Concatenamos como antes
df_global = []

for ric, (df, cols) in dfs.items():
    df = df.copy().reset_index()
    df['crypto'] = ric
    df.rename(columns={ric: 'close'}, inplace=True)

    # Renombrar columnas tipo 'BTCUSDT_lag_1' -> 'close_lag_1'
    lag_cols = {f'{ric}_lag_{i}': f'close_lag_{i}' for i in range(1, lags + 1)}
    df.rename(columns=lag_cols, inplace=True)

    df_global.append(df)

# Concatenar todo en un solo DataFrame
df_global = pd.concat(df_global, ignore_index=True)

# Ordenar por fecha
df_global = df_global.sort_values(by='timestamp').reset_index(drop=True)

# Vista rápida
print(df_global.head())


    timestamp        close         r           sma          min          max  \
0  2020-10-27  13636.17000  0.043770  11554.184333  10542.06000  13636.17000   
1  2020-10-27      0.25273  0.018490      0.247623      0.23273      0.25709   
2  2020-10-27      4.12170 -0.006361      4.083453      3.44200      4.48060   
3  2020-10-28  13266.40000 -0.027491  11639.860333  10542.06000  13636.17000   
4  2020-10-28      0.24540 -0.029432      0.247767      0.23273      0.25709   

        mom       vol        rsi         atr  ...  rsi_lag_2  rsi_lag_3  \
0  0.265626  0.017916  77.273397  166.124097  ...  74.124078  75.459534   
1  0.038588  0.018629  57.099688    0.003870  ...  57.840898  59.440599   
2 -0.113308  0.054953  42.733038    0.235048  ...  42.892232  43.088247   
3  0.240300  0.018860  71.765136  172.912293  ...  74.256884  74.124078   
4  0.017962  0.019369  53.598651    0.003985  ...  55.317583  57.840898   

   rsi_lag_4  rsi_lag_5   atr_lag_1   atr_lag_2   atr_lag_3   atr_la

In [27]:
df_global.columns

Index(['timestamp', 'close', 'r', 'sma', 'min', 'max', 'mom', 'vol', 'rsi',
       'atr', 'd', 'ten', 'close_lag_1', 'close_lag_2', 'close_lag_3',
       'close_lag_4', 'close_lag_5', 'r_lag_1', 'r_lag_2', 'r_lag_3',
       'r_lag_4', 'r_lag_5', 'ten_lag_1', 'ten_lag_2', 'ten_lag_3',
       'ten_lag_4', 'ten_lag_5', 'sma_lag_1', 'sma_lag_2', 'sma_lag_3',
       'sma_lag_4', 'sma_lag_5', 'min_lag_1', 'min_lag_2', 'min_lag_3',
       'min_lag_4', 'min_lag_5', 'max_lag_1', 'max_lag_2', 'max_lag_3',
       'max_lag_4', 'max_lag_5', 'mom_lag_1', 'mom_lag_2', 'mom_lag_3',
       'mom_lag_4', 'mom_lag_5', 'vol_lag_1', 'vol_lag_2', 'vol_lag_3',
       'vol_lag_4', 'vol_lag_5', 'rsi_lag_1', 'rsi_lag_2', 'rsi_lag_3',
       'rsi_lag_4', 'rsi_lag_5', 'atr_lag_1', 'atr_lag_2', 'atr_lag_3',
       'atr_lag_4', 'atr_lag_5', 'crypto'],
      dtype='object')

Hacemos una función que entrene el modelo, lo valide utilizando walk-forward y calcule el accuracy.

In [28]:
# Prueba a entrenar sin la columna d_lag_n para ver la importancia que tiene
'''
X_train = train.drop(columns=['d', 'timestamp'] + [col for col in train.columns if col.startswith('d_lag_')])
y_train = train['d']
X_test = test.drop(columns=['d', 'timestamp'] + [col for col in test.columns if col.startswith('d_lag_')])
y_test = test['d']
'''

"\nX_train = train.drop(columns=['d', 'timestamp'] + [col for col in train.columns if col.startswith('d_lag_')])\ny_train = train['d']\nX_test = test.drop(columns=['d', 'timestamp'] + [col for col in test.columns if col.startswith('d_lag_')])\ny_test = test['d']\n"

Modelo MLP Classifier GLOBAL

In [ ]:
def walk_forward_fit_test(model_class, data, freq, model_params={}, n_trials=5):
    if freq == '1h':
        period = pd.Timedelta(days=7)
    elif freq == '4h':
        period = pd.Timedelta(days=15)
    else:
        period = pd.Timedelta(days=90)
    final_test_period = pd.Timedelta(days=365)

    def normalize_with_close(X, close_col):
        ratio_cols = [col for col in X.columns if any(x in col for x in ['sma', 'atr', 'min', 'max'])]
        for col in ratio_cols:
            X[col] = X[col] / close_col
        return X

    def prepare_features(df):
        df = df.copy()
        crypto_dummies = pd.get_dummies(df['crypto'], prefix='crypto')
        df = pd.concat([df.drop(columns=['crypto']), crypto_dummies], axis=1)
        return df, crypto_dummies.columns

    def objective(trial):
        trial_params = {
            "hidden_layer_sizes": (trial.suggest_int("hidden_units", 32, 1024, step=64),),
            "alpha": trial.suggest_loguniform("alpha", 1e-5, 1e-1),
            "learning_rate_init": trial.suggest_loguniform("learning_rate", 1e-4, 1e-1),
            "max_iter": model_params.get("max_iter", 1000),
            "early_stopping": model_params.get("early_stopping", True),
            "validation_fraction": model_params.get("validation_fraction", 0.15),
            "shuffle": model_params.get("shuffle", False),
            "random_state": model_params.get("random_state", 100),
        }

        df = data.copy()
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df['d'] = (df['close'].shift(-window_pred) > df['close']).astype(int)
        df.dropna(inplace=True)

        max_time = df['timestamp'].max()
        cutoff = max_time - final_test_period
        df_trainval = df[df['timestamp'] < cutoff]

        min_time = df_trainval['timestamp'].min()
        split_dates = []
        current_time = min_time + period
        while current_time < cutoff:
            split_dates.append(current_time)
            current_time += period
        split_dates = split_dates[-5:]

        results = []
        crypto_accs = {}

        for split_date in split_dates:
            train = df_trainval[df_trainval['timestamp'] < (split_date - pd.Timedelta(days=window_pred))]
            test = df_trainval[(df_trainval['timestamp'] >= split_date) & (df_trainval['timestamp'] < split_date + period)]
            '''fin_train = split_date - pd.Timedelta(days=window_pred)
            fin_test= split_date + period
            print('split_date, comienzo test', split_date)            
            print('fin_train', fin_train)
            print('fin_test', fin_test )
            print('\n')'''

            if len(test) == 0:
                continue

            drop_cols = ['d'] + [col for col in train.columns if 'close' in col]
            X_train_raw, y_train = train.drop(columns=drop_cols), train['d']
            X_test_raw, y_test = test.drop(columns=drop_cols), test['d']

            # Guardar criptos antes de codificar
            cryptos_test = test['crypto'].values

            # Normalizar con close
            close_train, close_test = train['close_lag_1'], test['close_lag_1']
            X_train_raw = normalize_with_close(X_train_raw.copy(), close_train)
            X_test_raw = normalize_with_close(X_test_raw.copy(), close_test)

            # One-hot encoding
            X_train, _ = prepare_features(X_train_raw)
            X_test, _ = prepare_features(X_test_raw)

            mean, std = X_train.mean(), X_train.std()
            std.replace(0, 1, inplace=True)
            X_train = (X_train - mean) / std
            X_test = (X_test - mean) / std

            model = model_class(**trial_params)
            model.fit(X_train, y_train)

            # Importancia de cada característica
            '''result = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=0)
            sorted_idx = result.importances_mean.argsort()[::-1]
            print("Feature importances (top 10):")
            for i in sorted_idx[:10]:
                print(f"{X_train.columns[i]:<30} - Importance: {result.importances_mean[i]:.4f}")'''

            pred = np.where(model.predict(X_test) > 0.5, 1, 0)
            acc = accuracy_score(y_test, pred)
            results.append(acc)

            df_results_test = X_test.copy()
            df_results_test['true'] = y_test.values
            df_results_test['pred'] = pred
            df_results_test['crypto'] = cryptos_test

            accuracy_per_crypto = df_results_test.groupby('crypto').apply(lambda g: accuracy_score(g['true'], g['pred']))
            for crypto, acc_c in accuracy_per_crypto.items():
                crypto_accs.setdefault(crypto, []).append(acc_c)

        avg_acc = np.mean(results)
        print(f'VALIDATION | acc={avg_acc:.4f}')

        print("\nAccuracy promedio por criptomoneda (VAL):")
        for crypto, acc_list in crypto_accs.items():
            avg_crypto_acc = np.mean(acc_list)
            print(f"{crypto:<15} | acc = {avg_crypto_acc:.4f}")
            save_results_global(model_class.__name__, crypto, avg_crypto_acc, "Val", frequency=freq)

        save_results_global(model_class.__name__, "global", avg_acc, "Val", frequency=freq)
        return avg_acc

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, n_jobs=-1)

    best_params = study.best_params
    print("Mejores parámetros encontrados:", best_params)

    # Entrenamiento final con los mejores parámetros
    df = data.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['d'] = (df['close'].shift(-window_pred) > df['close']).astype(int)
    df.dropna(inplace=True)

    max_time = df['timestamp'].max()
    cutoff = max_time - final_test_period

    train = df[df['timestamp'] < (cutoff - pd.Timedelta(days=window_pred))]
    test = df[df['timestamp'] >= cutoff]

    if len(test) == 0:
        return best_params

    drop_cols = ['d'] + [col for col in train.columns if 'close' in col]
    X_train_raw, y_train = train.drop(columns=drop_cols), train['d']
    X_test_raw, y_test = test.drop(columns=drop_cols), test['d']
    cryptos_test = test['crypto'].values

    # Normalizar
    close_train, close_test = train['close_lag_1'], test['close_lag_1']
    X_train_raw = normalize_with_close(X_train_raw.copy(), close_train)
    X_test_raw = normalize_with_close(X_test_raw.copy(), close_test)

    # One-hot encoding
    X_train, _ = prepare_features(X_train_raw)
    X_test, _ = prepare_features(X_test_raw)

    mean, std = X_train.mean(), X_train.std()
    std.replace(0, 1, inplace=True)
    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std

    model = model_class(
        hidden_layer_sizes=(best_params["hidden_units"],),
        alpha=best_params["alpha"],
        learning_rate_init=best_params["learning_rate"],
        max_iter=model_params.get("max_iter", 1000),
        early_stopping=model_params.get("early_stopping", True),
        validation_fraction=model_params.get("validation_fraction", 0.15),
        shuffle=model_params.get("shuffle", False),
        random_state=model_params.get("random_state", 100),
    )
    model.fit(X_train, y_train)

    pred = np.where(model.predict(X_test) > 0.5, 1, 0)
    acc = accuracy_score(y_test, pred)
    print(f'FINAL TEST | acc={acc:.4f}')
    save_results_global(model_class.__name__, "global", acc, "Test", frequency=freq)

    df_results_test = X_test.copy()
    df_results_test['true'] = y_test.values
    df_results_test['pred'] = pred
    df_results_test['crypto'] = cryptos_test
    accuracy_per_crypto = df_results_test.groupby('crypto').apply(lambda g: accuracy_score(g['true'], g['pred']))
    print("\nFINAL TEST - Accuracy por criptomoneda:")
    for crypto, acc_c in accuracy_per_crypto.items():
        print(f"{crypto:<15} | acc = {acc_c:.4f}")
        save_results_global(model_class.__name__, crypto, acc_c, "Test", frequency=freq)

    return best_params


In [ ]:
      
# Ejecutar la optimización
model_params = {
    "max_iter": 1000,
    "early_stopping": True,
    "validation_fraction": 0.15,
    "shuffle": False,
    "random_state": 100
}

tuned_params = walk_forward_fit_test(MLPClassifier, df_global, frequency, model_params, n_trials=5)


[I 2025-05-04 20:09:28,077] A new study created in memory with name: no-name-133dcce8-7923-4322-b920-865611858725
C:\Users\raque\AppData\Local\Temp\ipykernel_4120\2666720479.py:13: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "alpha": trial.suggest_loguniform("alpha", 1e-5, 1e-1),
C:\Users\raque\AppData\Local\Temp\ipykernel_4120\2666720479.py:14: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "learning_rate_init": trial.suggest_loguniform("learning_rate", 1e-4, 1e-1),
[I 2025-05-04 20:09:31,826] Trial 0 finished with value: 0.5804444444444445 and parameters: {'hidden_units': 352, 'alpha': 0.0007812538210583971, 'learning_rate': 0.06249306265743307}. Best is trial 0 with

VALIDATION | acc=0.5804
Mejores parámetros encontrados: {'hidden_units': 352, 'alpha': 0.0007812538210583971, 'learning_rate': 0.06249306265743307}
Feature importances (top 10):
min_lag_1                      - Importance: 0.0192
min                            - Importance: 0.0182
min_lag_3                      - Importance: 0.0134
min_lag_4                      - Importance: 0.0133
min_lag_2                      - Importance: 0.0109
r_lag_5                        - Importance: 0.0105
vol                            - Importance: 0.0088
rsi_lag_1                      - Importance: 0.0081
ten_lag_2                      - Importance: 0.0073
min_lag_5                      - Importance: 0.0071
FINAL TEST | acc=0.6430
